# Studio OS — Wan 2.2 Colab Launcher
Select a GPU runtime (Runtime → Change runtime type), then Run all.
Picks the Wan 2.2 variant that fits the GPU, clones only `studio-os`, downloads weights, and generates one test video.

Needs: A100 80GB for the A14B model (~130GB download). L4/T4-class 24GB GPUs use the 5B model (~20GB download).


In [ ]:
!nvidia-smi
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f"GPU VRAM: {vram_gb:.1f} GiB")
if vram_gb >= 70:
    TASK, REPO, SIZE, EXTRA = "t2v-A14B", "Wan-AI/Wan2.2-T2V-A14B", "1280*720", ""
elif vram_gb >= 20:
    TASK, REPO, SIZE, EXTRA = "ti2v-5B", "Wan-AI/Wan2.2-TI2V-5B", "1280*704", "--t5_cpu"
else:
    raise SystemExit("No suitable GPU (need at least 24GB). Change runtime to a GPU with more memory.")
print("Selected:", TASK, REPO, SIZE)


In [ ]:
!git clone https://github.com/rayeshiacross-collab/studio-os.git /content/studio-os


In [ ]:
!git clone https://github.com/Wan-Video/Wan2.2.git /content/Wan2.2
%cd /content/Wan2.2
!grep -v flash_attn requirements.txt > /content/req-noflash.txt
!pip install -q -r /content/req-noflash.txt
!pip install -q "huggingface_hub[cli]"


In [ ]:
!huggingface-cli download {REPO} --local-dir /content/ckpt/{TASK}


In [ ]:
PROMPT = "A calm ocean at sunset, gentle waves, cinematic lighting"
OUT = "/content/wan22_test.mp4"
!python generate.py --task {TASK} --size {SIZE} --ckpt_dir /content/ckpt/{TASK} --offload_model True --convert_model_dtype {EXTRA} --prompt "{PROMPT}" --save_file {OUT}


In [ ]:
from IPython.display import Video
Video("/content/wan22_test.mp4", embed=True)
